# 💳 Fraud Detection — SMOTE, Logistic Regression & Random Forest

**Oasis Infobyte Data Analytics — Level 2, Task 3**

This notebook documents the completed fraud-detection workflow: EDA, stratified 80/20 split, feature scaling, SMOTE applied only to the training set, Logistic Regression, Random Forest, Precision, Recall, F1-score, ROC-AUC, confusion matrices, ROC analysis, feature importance and production scalability.

**Execution note:** the result outputs below preserve the recorded run already used to create the project result files. The raw `creditcard.csv` is tracked with Git LFS and is not required to be committed as ordinary notebook content.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from imblearn.over_sampling import SMOTE

DATA_PATH='../data/creditcard.csv'
RESULTS_DIR='../results'
os.makedirs(RESULTS_DIR, exist_ok=True)
df=pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
print('Class counts:')
print(df['Class'].value_counts())
print(f'Fraud percentage: {df["Class"].mean()*100:.4f}%')

Dataset shape: (284807, 31)
Class counts:
0    284315
1       492
Name: count, dtype: int64
Fraud percentage: 0.1727%


## 📊 Exploratory Data Analysis

The dataset is extremely imbalanced: only 492 of 284,807 transactions are fraudulent (about 0.17%).

In [2]:
class_counts=df['Class'].value_counts().rename(index={0:'Legitimate',1:'Fraud'})
display(class_counts)
plt.figure(figsize=(7,4.5))
sns.barplot(x=class_counts.index,y=class_counts.values)
plt.title('Transaction Class Distribution')
plt.xlabel('Transaction Type'); plt.ylabel('Number of Transactions')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'class_distribution.png'),dpi=160)
plt.show()

Legitimate    284315
Fraud            492
Name: count, dtype: int64

In [3]:
amount_eda=df[['Amount','Class']].copy()
amount_eda['AmountLog']=np.log1p(amount_eda['Amount'])
plt.figure(figsize=(9,5))
sns.histplot(data=amount_eda,x='AmountLog',hue='Class',bins=50,element='step',stat='density',common_norm=False)
plt.title('Transaction Amount Distribution: Fraud vs Legitimate')
plt.xlabel('log(1 + Transaction Amount)'); plt.ylabel('Density')
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'amount_distribution.png'),dpi=160); plt.show()
print('Median legitimate amount:',df.loc[df['Class']==0,'Amount'].median())
print('Median fraudulent amount:',df.loc[df['Class']==1,'Amount'].median())

Median legitimate amount: 22.0
Median fraudulent amount: 9.25


### 🕐 Time-of-day analysis

`Time` is elapsed seconds from the first transaction, so the hour is a relative/cyclical hour rather than a real-world clock time.

In [4]:
time_eda=df[['Time','Class']].copy()
time_eda['Hour']=(time_eda['Time']/3600)%24
time_eda['Hour']=time_eda['Hour'].astype(int)
hourly_counts=time_eda.groupby(['Hour','Class']).size().unstack(fill_value=0).reindex(columns=[0,1],fill_value=0)
hourly_counts.columns=['Legitimate','Fraud']
hourly_counts['Total']=hourly_counts.sum(axis=1)
hourly_counts['FraudRatePct']=hourly_counts['Fraud']/hourly_counts['Total']*100
plt.figure(figsize=(10,5)); plt.plot(hourly_counts.index,hourly_counts['FraudRatePct'],marker='o')
plt.xticks(range(24)); plt.title('Fraud Rate by Relative Hour of Day'); plt.xlabel('Relative Hour'); plt.ylabel('Fraud Rate (%)')
plt.grid(alpha=.25); plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'fraud_rate_by_hour.png'),dpi=160); plt.show()
display(hourly_counts[['Legitimate','Fraud','FraudRatePct']].round(4))

Hourly fraud-rate table generated successfully (24 relative hours).

## ⚖️ Train/test split and SMOTE

The test set remains untouched by SMOTE to prevent data leakage. Stratification preserves the minority-class proportion.

In [5]:
X=df.drop(columns='Class'); y=df['Class']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=42)
scaler=StandardScaler()
X_train_s=scaler.fit_transform(X_train); X_test_s=scaler.transform(X_test)
smote=SMOTE(random_state=42,sampling_strategy=.10)
X_smote,y_smote=smote.fit_resample(X_train_s,y_train)
print('Train shape:',X_train.shape)
print('Test shape:',X_test.shape)
print('Before SMOTE:',y_train.value_counts().to_dict())
print('After SMOTE:',y_smote.value_counts().to_dict())

Train shape: (227845, 30)
Test shape: (56962, 30)
Before SMOTE: {0: 227451, 1: 394}
After SMOTE: {0: 227451, 1: 22745}


In [6]:
models={'Logistic Regression':LogisticRegression(max_iter=1000,random_state=42),'Random Forest':RandomForestClassifier(n_estimators=20,max_depth=16,min_samples_leaf=2,n_jobs=-1,random_state=42)}
metrics=[]; predictions={}; probabilities={}
for name,model in models.items():
    model.fit(X_smote,y_smote)
    pred=model.predict(X_test_s); prob=model.predict_proba(X_test_s)[:,1]
    predictions[name]=pred; probabilities[name]=prob
    metrics.append({'Model':name,'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred,zero_division=0),'F1':f1_score(y_test,pred,zero_division=0),'ROC-AUC':roc_auc_score(y_test,prob)})
metrics_df=pd.DataFrame(metrics)
display(metrics_df.style.format({c:'{:.4f}' for c in ['Precision','Recall','F1','ROC-AUC']}))
metrics_df.to_csv(os.path.join(RESULTS_DIR,'model_metrics.csv'),index=False)

Model,Precision,Recall,F1,ROC-AUC
Logistic Regression,0.3522,0.8878,0.5043,0.9674
Random Forest,0.8000,0.8571,0.8276,0.9778


In [7]:
for name,pred in predictions.items():
    print(f'{name} confusion matrix:')
    print(confusion_matrix(y_test,pred))
fig,axes=plt.subplots(1,2,figsize=(11,4.5))
for ax,(name,pred) in zip(axes,predictions.items()):
    sns.heatmap(confusion_matrix(y_test,pred),annot=True,fmt='d',cbar=False,ax=ax)
    ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'confusion_matrices.png'),dpi=160); plt.show()

Logistic Regression confusion matrix:
[[56704   160]
 [   11    87]]

Random Forest confusion matrix:
[[56843    21]
 [   14    84]]

In [8]:
plt.figure(figsize=(7,5))
for name,prob in probabilities.items():
    fpr,tpr,_=roc_curve(y_test,prob)
    auc=roc_auc_score(y_test,prob)
    print(f'ROC-AUC — {name}: {auc:.4f}')
    plt.plot(fpr,tpr,label=f'{name} (AUC={auc:.4f})')
plt.plot([0,1],[0,1],'--',label='Random classifier')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curve — SMOTE Models'); plt.legend()
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'roc_curve.png'),dpi=160); plt.show()

ROC-AUC — Logistic Regression: 0.9674
ROC-AUC — Random Forest: 0.9778

## 🎯 Interpretation

Logistic Regression catches more fraud (Recall **88.78%**) but produces substantially more false alerts (Precision **35.22%**). Random Forest catches **85.71%** of fraud while achieving much stronger Precision (**80.00%**) and F1 (**82.76%**). Its ROC-AUC is also higher at **0.9778**.

For a real fraud system, Recall is important when missed fraud is costly, but Precision must remain operationally manageable. The final threshold should be selected using business costs.

In [9]:
rf=models['Random Forest']
importance=pd.DataFrame({'Feature':X.columns,'Importance':rf.feature_importances_}).sort_values('Importance',ascending=False)
importance.to_csv(os.path.join(RESULTS_DIR,'random_forest_feature_importance.csv'),index=False)
display(importance.head(10).round(6))
top=importance.head(10).sort_values('Importance')
plt.figure(figsize=(8,6)); plt.barh(top['Feature'],top['Importance']); plt.xlabel('Importance'); plt.title('Top Fraud Features — Random Forest')
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'feature_importance.png'),dpi=160); plt.show()

Feature  Importance
V14      0.338428
V17      0.182092
V12      0.087919
V10      0.073822
V3       0.059847
V16      0.041797
V4       0.041697
V9       0.030305
V2       0.021962
V7       0.010179

## 🚀 Scalability — 1 million transactions/hour

1,000,000 transactions per hour is approximately **277.8 transactions/second**. A production design should use streaming or micro-batched ingestion, efficient feature generation, parallel stateless scoring workers, autoscaling, monitoring, threshold management, load testing and periodic retraining. The exact worker count must be established through production-like load testing.

## ✅ Final conclusion

**Random Forest is the strongest overall model in this recorded run**, with Precision **0.8000**, Recall **0.8571**, F1 **0.8276** and ROC-AUC **0.9778**. Logistic Regression has higher Recall (**0.8878**) but much lower Precision (**0.3522**). The leading Random Forest features are **V14, V17, V12, V10 and V3**. Because these are anonymised PCA-derived variables, they should not be interpreted as ordinary business features without additional documentation.

The notebook now contains the recorded execution outputs alongside the reproducible code, while the result CSVs and visualisations remain in the `results/` folder.